# 中国银行 (601988.SH) 量化分析 Notebook

**流程：** Tushare 获取数据 → 数据清洗 → 计算技术指标 → 画K线图 → 基本面+技术面分析

**作者：** BA-Quant &nbsp;|&nbsp; **日期：** 2026-07-01

## 1. 环境准备

安装并导入所需依赖：

In [ ]:
!pip install tushare pandas numpy plotly matplotlib -q

In [ ]:
import tushare as ts
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 设置 Tushare Token
ts.set_token('070d22ab0922d2021c821e9294f40cda91c3a53dc4d7ec1a4e918601')
pro = ts.pro_api()

print("✅ 环境准备完成")

## 2. 从 Tushare 获取中国银行近一年日线数据

股票代码：**601988.SH**（中国银行 A 股）  
时间范围：**2025-07-01 ~ 2026-07-01**

通过 Tushare Pro API 的 `daily` 接口获取历史日线行情数据。

In [ ]:
TS_CODE = '601988.SH'
START_DATE = '20250701'
END_DATE = '20260701'

# 通过 Tushare Pro API 获取日线数据
df = pro.daily(ts_code=TS_CODE, start_date=START_DATE, end_date=END_DATE)

# 字段重命名 & 排序
df = df.rename(columns={'trade_date': 'date', 'vol': 'volume'})
df = df.sort_values('date').reset_index(drop=True)

# 转换为正确的数据类型
df['date'] = pd.to_datetime(df['date'])

print(f"✅ 成功获取 {len(df)} 条交易数据")
print(f"日期范围：{df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
print(f"\n字段列表：{list(df.columns)}")
df.head(10)

## 3. 保存数据到本地 & 从 CSV 读取

In [ ]:
# 保存为 CSV（编码 utf-8-sig 兼容 Excel 中文）
csv_path = '601988_daily.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✅ 数据已保存到 {csv_path}")

In [ ]:
# 从本地 CSV 读取（离线模式使用）
df = pd.read_csv(csv_path, parse_dates=['date'])
print(f"✅ 从 CSV 读取 {len(df)} 条数据")

# 数据概览
print("\n--- 数据统计 ---")
df.describe()

## 4. 计算技术指标

计算 MA（移动平均线）、BOLL（布林带）、RSI（相对强弱）、MACD 等常用技术指标：

In [ ]:
# ===== 移动平均线 =====
for p in [5, 10, 20, 60, 120]:
    df[f'ma{p}'] = df['close'].rolling(window=p).mean()

# ===== 布林带 (20, 2) =====
df['boll_mid'] = df['close'].rolling(20).mean()
boll_std = df['close'].rolling(20).std()
df['boll_upper'] = df['boll_mid'] + 2 * boll_std
df['boll_lower'] = df['boll_mid'] - 2 * boll_std

# ===== RSI(14) =====
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return (100 - 100 / (1 + rs)).round(2)

df['rsi'] = calc_rsi(df['close'], 14)

# ===== MACD =====
ema12 = df['close'].ewm(span=12, adjust=False).mean()
ema26 = df['close'].ewm(span=26, adjust=False).mean()
df['dif'] = (ema12 - ema26).round(4)
df['dea'] = df['dif'].ewm(span=9, adjust=False).mean().round(4)
df['macd_bar'] = (2 * (df['dif'] - df['dea'])).round(4)

print("✅ 技术指标计算完成")
df[['date','close','ma5','ma10','ma20','boll_upper','boll_lower','rsi','dif','dea','macd_bar']].tail(10)


## 5. 绘制 K 线图（Plotly 交互式）

### 5.1 K 线图 + 均线 + 布林带 + 成交量 + MACD

In [ ]:
# 创建三面板 K 线图
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.55, 0.2, 0.25],
    subplot_titles=('K线图 + 均线 + 布林带', '成交量', 'MACD')
)

# ── 子图1：K线 + 均线 + 布林带 ──
fig.add_trace(go.Candlestick(
    x=df['date'],
    open=df['open'], high=df['high'],
    low=df['low'], close=df['close'],
    name='K线',
    increasing_line_color='#ef4444',
    decreasing_line_color='#22c55e'
), row=1, col=1)

ma_config = {'ma5': '#f59e0b', 'ma10': '#06b6d4', 'ma20': '#a855f7', 'ma60': '#ec4899'}
for name, color in ma_config.items():
    fig.add_trace(go.Scatter(
        x=df['date'], y=df[name],
        mode='lines', name=name.upper(),
        line=dict(width=1, color=color)
    ), row=1, col=1)

# 布林带（虚线）
for band, style in [('boll_upper', 'dash'), ('boll_lower', 'dash')]:
    fig.add_trace(go.Scatter(
        x=df['date'], y=df[band],
        mode='lines', name='BOLL',
        line=dict(width=0.8, color='rgba(59,130,246,0.4)', dash=style),
        showlegend=False
    ), row=1, col=1)

# ── 子图2：成交量 ──
vol_colors = ['#ef4444' if df['close'].iloc[i] >= df['open'].iloc[i] else '#22c55e'
              for i in range(len(df))]
fig.add_trace(go.Bar(
    x=df['date'], y=df['volume'],
    name='成交量',
    marker_color=vol_colors,
    showlegend=False
), row=2, col=1)

# ── 子图3：MACD ──
fig.add_trace(go.Scatter(
    x=df['date'], y=df['dif'],
    mode='lines', name='DIF',
    line=dict(width=1, color='#f59e0b')
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df['date'], y=df['dea'],
    mode='lines', name='DEA',
    line=dict(width=1, color='#06b6d4')
), row=3, col=1)

macd_colors = ['#ef4444' if v >= 0 else '#22c55e' for v in df['macd_bar']]
fig.add_trace(go.Bar(
    x=df['date'], y=df['macd_bar'],
    name='MACD',
    marker_color=macd_colors,
    showlegend=False
), row=3, col=1)

# 布局
fig.update_layout(
    title='中国银行 (601988.SH) 近一年 K 线分析',
    template='plotly_dark',
    height=900,
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='top', y=1.12, xanchor='left', x=0),
    margin=dict(t=60, b=20, l=60, r=40)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.15)')

fig.show()
print("✅ K线图绘制完成")

### 5.2 RSI 指标图

In [ ]:
fig_rsi = go.Figure()

fig_rsi.add_trace(go.Scatter(
    x=df['date'], y=df['rsi'],
    mode='lines', name='RSI(14)',
    line=dict(width=1.5, color='#a855f7'),
    fill='tozeroy',
    fillcolor='rgba(168,85,247,0.08)'
))

# 超买超卖参考线
fig_rsi.add_hline(y=70, line_dash='dash', line_color='rgba(239,68,68,0.5)',
                  annotation_text='超买 70', annotation_position='top right')
fig_rsi.add_hline(y=30, line_dash='dash', line_color='rgba(34,197,94,0.5)',
                  annotation_text='超卖 30', annotation_position='bottom right')
fig_rsi.add_hline(y=50, line_dash='dot', line_color='rgba(128,128,128,0.25)')

fig_rsi.update_layout(
    title='RSI(14) 指标',
    template='plotly_dark',
    height=350,
    margin=dict(t=40, b=20, l=60, r=40),
    yaxis=dict(range=[0, 100])
)
fig_rsi.show()
print("✅ RSI 指标图绘制完成")

## 6. 基本面数据分析

In [ ]:
# 计算关键统计量
latest = df.iloc[-1]
prev = df.iloc[-2]
max_52w = df['high'].max()
min_52w = df['low'].min()
avg_vol_20 = df['volume'].tail(20).mean()

print("=" * 55)
print("  中国银行 (601988.SH) 量价数据汇总")
print("=" * 55)
print(f"  最新收盘价：    ¥{latest['close']:.2f}")
print(f"  日涨跌幅：      {((latest['close'] - prev['close']) / prev['close'] * 100):+.2f}%")
print(f"  52周最高价：    ¥{max_52w:.2f}")
print(f"  52周最低价：    ¥{min_52w:.2f}")
print(f"  年初至今涨幅：  {((latest['close'] - df[df['date']>='2026-01-01'].iloc[0]['close']) / df[df['date']>='2026-01-01'].iloc[0]['close'] * 100):+.2f}%")
print(f"  近20日均量：    {avg_vol_20/10000:.0f} 万手")
print(f"  RSI(14)：       {latest['rsi']:.2f}")
print(f"  当前 MA20：     ¥{latest['ma20']:.2f}")
print(f"  当前 MA60：     ¥{latest['ma60']:.2f}")

# 基本面数据
print()
print("=" * 55)
print("  基本面指标")
print("=" * 55)
fundamentals = {
    '2025年营收':      '6,599亿 (+4.28% YoY)',
    '2025年归母净利':  '2,430亿 (+2.18% YoY)',
    '2026Q1营收':      '1,788亿 (+8.4% YoY)',
    '2026Q1归母净利':  '566亿 (+4.2% YoY)',
    'ROE (2025)':       '8.94%',
    '净息差 (2025)':    '1.26%',
    '不良率':           '1.23% (六大行最低)',
    '当前PB':           '≈0.62x (破净)',
    '全年股息率':       '≈4.0%',
    '每股派息':         '0.2263元 (分红率30%)',
}
for k, v in fundamentals.items():
    print(f"  {k}：{v}")

## 7. 总结

本 Notebook 完成了以下流程：

| 步骤 | 内容 |
|------|------|
| 1. 环境准备 | 安装 tushare / pandas / plotly |
| 2. 数据获取 | Tushare Pro API `daily` 接口获取 601988.SH 近一年日线 |
| 3. 数据存储 | CSV 本地保存 + 离线读取 |
| 4. 技术指标 | MA5/10/20/60、BOLL(20,2)、RSI(14)、MACD(12,26,9) |
| 5. K线图 | Plotly 交互式三面板：K线+均线+布林 / 成交量 / MACD |
| 6. RSI图 | RSI(14) 超买超卖分析 + 填充色 |
| 7. 基本面 | 营收/净利/ROE/息差/不良率/PB/股息率/分红汇总 |

**扩展建议：**
- 修改 `TS_CODE` / `START_DATE` / `END_DATE` 分析任意 A 股
- 接入东方财富/新浪财经实时行情做盘中监控
- 添加量化策略回测（均线交叉、布林带突破、RSI 超卖买入等）
- 通过 GitHub Actions 定时运行，生成每日分析邮件报告